# Packages import

In [1]:
import os
import json
import time
import requests
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup

# Ceneo scraper

1. Provide URL address of products opinions webpage

In [2]:

product_code = '100714868'
page = 1
url = f'https://www.ceneo.pl/{product_code}/opinie-{page}'

2. Send the request to provide url address

In [3]:
path_to_driver = "D:\\chromedriver-win64\\chromedriver.exe"
service = Service(path_to_driver)
driver = webdriver.Chrome(service=service)
driver.get(url)
time.sleep(10)
driver.find_element(by='xpath', value='/html/body/div[2]/div[2]/div/div/div[1]/div/div[2]/button[1]').click()

InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=147.0.7727.138); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff6bea337c5+15b35]
	chromedriver!GetHandleVerifier [0x7ff6bea33830+15ba0]
	chromedriver!(No symbol) [0x7ff6be77d8dd]
	chromedriver!(No symbol) [0x7ff6be768a72]
	chromedriver!(No symbol) [0x7ff6be78f1b6]
	chromedriver!(No symbol) [0x7ff6be807740]
	chromedriver!(No symbol) [0x7ff6be8246c2]
	chromedriver!(No symbol) [0x7ff6be7c9348]
	chromedriver!(No symbol) [0x7ff6be7ca233]
	chromedriver!GetHandleVerifier [0x7ff6bed25e39+3081a9]
	chromedriver!GetHandleVerifier [0x7ff6bed2054a+3028ba]
	chromedriver!GetHandleVerifier [0x7ff6bed417f2+323b62]
	chromedriver!GetHandleVerifier [0x7ff6bea50bd5+32f45]
	chromedri

In [42]:
response = requests.get(url)
print(response.status_code)

200


3. If status code is OK, fetch product name

In [43]:
page_dom = BeautifulSoup(response.text, 'html.parser')
print(type(page_dom))

<class 'bs4.BeautifulSoup'>


In [44]:
product_name = page_dom.select_one("h1.product-top__product-info__name").get_text()
print(product_name)

AttributeError: 'NoneType' object has no attribute 'get_text'

4. If status code is OK, fetch all opinions from requested webpage

In [6]:
opinions = page_dom.select("div.js_product-review:not(.user-post--highlight)")
print(type(opinions))
print(len(opinions))

<class 'bs4.element.ResultSet'>
10


In [7]:
all_opinions = page_dom.find_all('div', class_='js_product-review')
opinions = [r for r in all_opinions if 'user-post--highlight' not in r.get('class', [])]
print(type(opinions))
print(len(opinions))

<class 'list'>
10


5. For all fetched opinions, parse them to extract relevant data

In [8]:
all_opinions = []
for opinion in opinions:
    single_opinion = {
        'opinion_id': opinion['data-entry-id'],
        'author': opinion.select_one('span.user-post__author-name').get_text().strip(),
        'recommendation': opinion.select_one('span.user-post__author-recomendation > em').get_text().strip(),
        'score': opinion.select_one('span.user-post__score-count').get_text().strip(),
        'content': opinion.select_one('div.user-post__text').get_text().strip(),
        'pros': [p.get_text() for p in opinion.select('div.review-feature__item--positive')],
        'cons': [c.get_text() for c in opinion.select('div.review-feature__item--negative')],
        'like': opinion.select_one('span[id^="votes-yes"]').get_text().strip(),
        'dislike': opinion.select_one('button.vote-no > span').get_text().strip(),
        'publishing_date': opinion.select_one('span.user-post__published > time:nth-child(1)')['datetime'],
        'purchase_date': opinion.select_one('span.user-post__published > time:nth-child(2)')['datetime'].strip() if opinion.select_one('span.user-post__published > time:nth-child(2)[datetime]') else None
    }
    all_opinions.append(single_opinion)


6. Check if there is next page with opinions 

In [9]:
next = True if page_dom.select_one('button.pagination__next') else False
if next: page +=1


8. Save obtained opinions

In [10]:
if not os.path.exists("./opinions"):
    os.mkdir("./opinions")

In [11]:
with open(f"./opinions/{product_code}.json", "w", encoding="UTF-8") as jf:
    json.dump(all_opinions, jf, indent=4,ensure_ascii=False)
